# CATSS-based Greek correction pipeline

Corrects `hebrew_words[*].greek` in the per-chapter bible JSONs using the
CCAT/CATSS parallel Hebrew–Greek database (as rendered `.par.html` files in
`~/m_penn/out1/`).

**Two modes** controlled by `APPLY_CATSS_CHANGES`:
- `0` → report only. Counts what would change, prints verse-level diffs.
- `1` → also write back the condensed JSON, using the same compact
  serializer as `4correct_json.ipynb`.

This notebook only touches **Greek** (`hebrew_words[*].greek` and
`leftover_greek`). It never touches Hebrew or Latvian — those are owned
by `4correct_json.ipynb` and should be run first.


## config

In [ ]:
APPLY_CATSS_CHANGES = 0   # 0 = report only, 1 = write back
verbose = 1               # per-verse diff printing
verbose_skipped = 1       # print verses where we couldn't make any fix
PAR_DIR = "~/m_penn/out1"  # folder with the 01.Genesis.par.html … files
BIBLE_DIR = "bible"        # folder with per-book per-chapter json files

# When ON, any word currently in a JSON verse's leftover_greek whose
# normalized key does NOT appear anywhere in that verse's CATSS content
# (neither mapped nor leftover) is removed. This cleans up the residual
# "wrong-verse leftovers" that the previous AI pipeline created in books
# where LXX and MT use different verse numbering (1 Samuel 24, Daniel 3-4,
# Numbers 17, Psalms titles, etc.). The correct verse will get those words
# from its own CATSS pass.
CATSS_AUTHORITATIVE_LEFTOVERS = 1

# For duplicated CATSS books (JoshuaA/B, JudgesA/B, DanielOG/Th) we run ALL
# of them — later passes only apply safe fixes on top of the earlier pass,
# so no information is lost and any extra agreement from the second tradition
# just gives more chances to correct AI-made mappings.


## imports + CATSS → bible book name mapping

In [ ]:
import os, re, json, html as html_mod
from pathlib import Path
from collections import Counter, defaultdict
from tqdm.notebook import tqdm

PAR_DIR_P = Path(os.path.expanduser(PAR_DIR))

# CATSS filename stem  →  bible/ folder name
# Books with no corresponding MT Hebrew folder (Ps151, 1Esdras, Sirach, Baruch)
# are mapped to None and skipped.
CATSS_TO_BIBLE = {
    "01.Genesis":   "genesis",
    "02.Exodus":    "exodus",
    "03.Lev":       "leviticus",
    "04.Num":       "numbers",
    "05.Deut":      "deuteronomy",
    "06.JoshB":     "joshua",
    "07.JoshA":     "joshua",
    "08.JudgesB":   "judges",
    "09.JudgesA":   "judges",
    "10.Ruth":      "ruth",
    "11.1Sam":      "1_samuel",
    "12.2Sam":      "2_samuel",
    "13.1Kings":    "1_kings",
    "14.2Kings":    "2_kings",
    "15.1Chron":    "1_chronicles",
    "16.2Chron":    "2_chronicles",
    "17.1Esdras":   None,   # deuterocanonical, no MT
    "18.Esther":    "esther",
    "18.Ezra":      "ezra",
    "19.Neh":       "nehemiah",
    "20.Psalms":    "psalms",
    "22.Ps151":     None,   # deuterocanonical
    "23.Prov":      "proverbs",
    "24.Qoh":       "ecclesiastes",
    "25.Cant":      "songs",
    "26.Job":       "job",
    "27.Sirach":    None,   # deuterocanonical
    "28.Hosea":     "hosea",
    "29.Micah":     "micah",
    "30.Amos":      "amos",
    "31.Joel":      "joel",
    "32.Jonah":     "jonah",
    "33.Obadiah":   "obadiah",
    "34.Nahum":     "nahum",
    "35.Hab":       "habakkuk",
    "36.Zeph":      "zephaniah",
    "37.Haggai":    "haggai",
    "38.Zech":      "zechariah",
    "39.Malachi":   "malachi",
    "40.Isaiah":    "isaiah",
    "41.Jer":       "jeremiah",
    "42.Baruch":    None,   # deuterocanonical
    "43.Lam":       "lamentations",
    "44.Ezekiel":   "ezekiel",
    "45.DanielOG":  "daniel",
    "46.DanielTh":  "daniel",
}

# Inverse: bible folder → list of CATSS stems (in the order we want to apply)
# DanielTh first (matches MT chapter structure more closely), then DanielOG
# as a second-opinion pass. JoshB/JudgesB first (LXX mainstream), then A.
BIBLE_TO_CATSS = defaultdict(list)
_order_hint = {
    "joshua": ["06.JoshB", "07.JoshA"],
    "judges": ["08.JudgesB", "09.JudgesA"],
    "daniel": ["46.DanielTh", "45.DanielOG"],
}
for stem, bible in CATSS_TO_BIBLE.items():
    if bible is None:
        continue
    BIBLE_TO_CATSS[bible].append(stem)
for bible, ordered in _order_hint.items():
    if bible in BIBLE_TO_CATSS:
        BIBLE_TO_CATSS[bible] = ordered

print(f"PAR_DIR resolved to: {PAR_DIR_P}")
print(f"{len(BIBLE_TO_CATSS)} bible folders covered by CATSS")


## Verse-number remapping (MT JSON ↔ CATSS)

CATSS `.par.html` files use **English/LXX** chapter-verse numbering for
several books (1 Samuel 23/24 boundary, Numbers 17, Daniel 3–4, Psalms
titles, etc.), while our JSONs use **MT** numbering.

The mapping is **verbatim copied** from cell 7 of `4correct_json.ipynb`
(`l65_to_hb` / `l65_to_hb_merge`) because the Latvian 1965 Bible uses the
same English verse boundaries that CATSS uses. If you update the table in
`4correct_json.ipynb`, also update it here.

Merge/split cases (marked with `"!"` placeholder in the original derivation)
are handled as **skip** in this notebook — we don't try to split CATSS cells
across a JSON merged verse, because that would require fragile text
alignment. Those verses are reported and left untouched.


In [ ]:
#split strings ir pirmaaas daljas nobeigums, ieskaitot, naakamaa dalja ir peec taa ko noraada
# l65_to_hb_merge[(('1_chronicles', 12, 4), "trīsdesmit")] = (('1_chronicles', 12, 4), ('1_chronicles', 12, 5))
# pirmaa dala ietvers sevii visu liidz vaardam triisdesmit ieskaitot (12,4), otraa tuuliit aiz taa triisdesmit(12,5)
# lv vietturis : ebreju vietturis
l65_to_hb_merge = {}
l65_to_hb={
('1_chronicles', 5, 27) : ('1_chronicles', 6, 1),
('1_chronicles', 5, 28) : ('1_chronicles', 6, 2),
('1_chronicles', 5, 29) : ('1_chronicles', 6, 3),
('1_chronicles', 5, 30) : ('1_chronicles', 6, 4),
('1_chronicles', 5, 31) : ('1_chronicles', 6, 5),
('1_chronicles', 5, 32) : ('1_chronicles', 6, 6),
('1_chronicles', 5, 33) : ('1_chronicles', 6, 7),
('1_chronicles', 5, 34) : ('1_chronicles', 6, 8),
('1_chronicles', 5, 35) : ('1_chronicles', 6, 9),
('1_chronicles', 5, 36) : ('1_chronicles', 6, 10),
('1_chronicles', 5, 37) : ('1_chronicles', 6, 11),
('1_chronicles', 5, 38) : ('1_chronicles', 6, 12),
('1_chronicles', 5, 39) : ('1_chronicles', 6, 13),
('1_chronicles', 5, 40) : ('1_chronicles', 6, 14),
('1_chronicles', 5, 41) : ('1_chronicles', 6, 15)
}
for i in range(1, 66+1):
    l65_to_hb[('1_chronicles', 6, i)]=('1_chronicles', 6, i+15)

l65_to_hb_merge[(('1_chronicles', 12, 4), "trīsdesmit")] = (('1_chronicles', 12, 4), ('1_chronicles', 12, 5))
for i in range(5, 39+1):
    l65_to_hb[('1_chronicles', 12, i)] = ('1_chronicles', 12, i+1)
l65_to_hb_merge [(('1_chronicles', 12, 39), ('1_chronicles', 12, 40))] = ('1_chronicles', 12, 40)


for l, h in zip(range(1, 14+1), range(21, 34+1)):
    l65_to_hb[('1_kings', 5, l)]=('1_kings', 4, h)

for i in range(15, 32+1):
    l65_to_hb[('1_kings', 5, i)]=('1_kings', 5, i-14)



#merge 43 + 44
l65_to_hb_merge[(('1_kings', 22, 43), ('1_kings', 22, 44))]= ('1_kings', 22, 43)

#now update mapping
for i in range(43, 53+1):
    l65_to_hb[('1_kings', 22, i+1)]=('1_kings', 22, i)
#1 samuel 20:42+
l65_to_hb[('1_samuel', 20, 43)]=('1_samuel', 21, 1)
for i in range(1, 13+1):
    l65_to_hb[('1_samuel', 21, i)]=('1_samuel', 21, i+1)
#14, 15 merged => 14 hebrw
l65_to_hb_merge[(('1_samuel', 21, 14), ('1_samuel', 21, 15))] = ('1_samuel', 21, 15)

#1 samuel 23:28+
l65_to_hb[('1_samuel', 24, 1)] = ('1_samuel', 23, 29)
for i in range(2, 23+1):
    l65_to_hb[('1_samuel', 24, i)]=('1_samuel', 24, i-1)

#2chronicles 2:1
l65_to_hb[('2_chronicles', 1, 18)] = ('2_chronicles', 2, 1)
for i in range(1, 17+1):
    l65_to_hb[('2_chronicles', 2, i)]=('2_chronicles', 2, i+1)

#2chronicles 14:1
l65_to_hb[('2_chronicles', 13, 23)] = ('2_chronicles', 14, 1)
for i in range(1, 14+1):
    l65_to_hb[('2_chronicles', 14, i)]=('2_chronicles', 14, i+1)

#2kings 12:1-21
l65_to_hb[('2_kings', 12, 1)] = ('2_kings', 11, 21)
for i in range(2, 22+1):
    l65_to_hb[('2_kings', 12, i)]=('2_kings', 12, i-1)

#2 sam 18:33
l65_to_hb[('2_samuel', 19, 1)] = ('2_samuel', 18, 33)
for i in range(2, 44+1):
    l65_to_hb[('2_samuel', 19, i)]=('2_samuel', 19, i-1)

#daniel 3
l65_to_hb[('daniel', 3, 31)] = ('daniel', 4, 1)
#obsolote, now merge ir rghtfully updating the map.l65_to_hb[('daniel', 3, 32)] = ('daniel', 4, 2), 3

#!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!1 O T R Ā D I !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
# 1 lv verse = 2x heb verses, TODO: SPLIT THEM ON WORD BEING LAST IN 1st part
l65_to_hb_merge[(('daniel', 3, 32), "darījis")] = (('daniel', 4, 2), ('daniel', 4, 3))
for i in range(1, 34+1):
    l65_to_hb[('daniel', 4, i)] = ('daniel', 4, i+3)

#!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!1 O T R Ā D I !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
# 1 lv verse = 2x heb verses, TODO: SPLIT THEM ON WORD BEING LAST IN 1st part
l65_to_hb_merge[(('daniel', 5, 30), "nogalināts")] = (('daniel', 5, 30), ('daniel', 5, 31))

l65_to_hb[('deuteronomy', 13, 1)] = ('deuteronomy', 12, 32)
for i in range(2, 19+1):
    l65_to_hb[('deuteronomy', 13, i)]=('deuteronomy', 13, i-1)

l65_to_hb[('deuteronomy', 23, 1)] = ('deuteronomy', 22, 30)
for i in range(2, 26+1):
    l65_to_hb[('deuteronomy', 23, i)]=('deuteronomy', 23, i-1)

l65_to_hb[('deuteronomy', 28, 69)] = ('deuteronomy', 29, 1)
l65_to_hb_merge[(('deuteronomy', 29, 1), ('deuteronomy', 29, 2))]=('deuteronomy', 29, 2)

#ecclesiates
l65_to_hb[('ecclesiastes', 4, 17)] = ('ecclesiastes', 5, 1)
for i in range(1, 19+1):
    l65_to_hb[('ecclesiastes', 5, i)]=('ecclesiastes', 5, i+1)

l65_to_hb[('exodus', 1, 21)] = ('exodus', 1, 22)

l65_to_hb[('exodus', 7, 26)] = ('exodus', 8, 1)
l65_to_hb[('exodus', 7, 27)] = ('exodus', 8, 2)
l65_to_hb[('exodus', 7, 28)] = ('exodus', 8, 3)
l65_to_hb[('exodus', 7, 29)] = ('exodus', 8, 4)
for i in range(1, 28+1):
    l65_to_hb[('exodus', 8, i)]=('exodus', 8, i+4)

l65_to_hb[('exodus', 21, 37)] = ('exodus', 22, 1)
for i in range(1, 30+1):
    l65_to_hb[('exodus', 22, i)]=('exodus', 22, i+1)
#ezekiel
for i in range(45, 49+1):
    l65_to_hb[('ezekiel', 21, i-44)]=('ezekiel', 20, i)
for i in range(6, 37+1):
    l65_to_hb[('ezekiel', 21, i)]=('ezekiel', 21, i-5)

# genesis 3 panti savienoti vienā un izlaista 1 tauta
l65_to_hb_merge[(('genesis', 15, 19), "kadmoniešus", "refajiešus")] = (('genesis', 15, 19), ('genesis', 15, 20), ('genesis', 15, 21))

l65_to_hb_merge[(('genesis', 24, 65),('genesis', 24, 66))] = ('genesis', 24, 65)
for i in range(67, 68+1):
    l65_to_hb[('genesis', 24, i)]=('genesis', 24, i-1)

l65_to_hb[('genesis', 32, 1)]=('genesis', 31, 55)
for i in range(2, 33+1):
    l65_to_hb[('genesis', 32, i)]=('genesis', 32, i-1)

#hosea 1
l65_to_hb[('hosea', 2, 1)] = ('hosea', 1, 10)
l65_to_hb[('hosea', 2, 2)] = ('hosea', 1, 11)
for i in range(3, 25+1):
    l65_to_hb[('hosea', 2, i)]=('hosea', 2, i-2)

l65_to_hb[('hosea', 12, 1)] = ('hosea', 11, 12)
for i in range(2, 15+1):
    l65_to_hb[('hosea', 12, i)]=('hosea', 12, i-1)

l65_to_hb[('hosea', 14, 1)] =('hosea', 13, 16)
for i in range(2, 10+1):
    l65_to_hb[('hosea', 14, i)]=('hosea', 14, i-1)

#isaiah
l65_to_hb[('isaiah', 8, 23)] = ('isaiah', 9, 1)
for i in range(1, 20+1):
    l65_to_hb[('isaiah', 9, i)]=('isaiah', 9, i+1)

l65_to_hb[('jeremiah', 8, 23)] = ('jeremiah', 9, 1)
for i in range(1, 25+1):
    l65_to_hb[('jeremiah', 9, i)]=('jeremiah', 9, i+1)

# job
for i in range(25, 32+1):
    l65_to_hb[('job', 40, i)]=('job', 41, i-24)
for i in range(1, 26+1):
    l65_to_hb[('job', 41, i)]=('job', 41, i+8)
#joel
for i in range(1, 5+1):
    l65_to_hb[('joel', 3, i)]=('joel', 2, i+27)
for i in range(1, 21+1):
    l65_to_hb[('joel', 4, i)]=('joel', 3, i)

#jonah
l65_to_hb[('jonah', 2, 1)] = ('jonah', 1, 17)
for i in range(2, 11+1):
    l65_to_hb[('jonah', 2, i)]=('jonah', 2, i-1)

#leviticus
for i in range(20, 26+1):
    l65_to_hb[('leviticus', 5, i)]=('leviticus', 6, i-19)
for i in range(1, 23+1):
    l65_to_hb[('leviticus', 6, i)]=('leviticus', 6, i+7)

#malachi
for i in range(19, 24+1):
    l65_to_hb[('malachi', 3, i)]=('malachi', 4, i-18)
#malachi 4 in Latvian does not exist!!

#micah
l65_to_hb[('micah', 4, 14)] = ('micah', 5, 1)
for i in range(1, 14+1):
    l65_to_hb[('micah', 5, i)]=('micah', 5, i+1)

#nahum
l65_to_hb[('nahum', 2, 1)] = ('nahum', 1, 15)
for i in range(2, 14+1):
    l65_to_hb[('nahum', 2, i)]=('nahum', 2, i-1)

#nehemiah
l65_to_hb[('nehemiah', 10, 1)] = ('nehemiah', 9, 38)
for i in range(2, 40+1):
    l65_to_hb[('nehemiah', 10, i)]=('nehemiah', 10, i-1)


#numbers
for i in range(36, 50+1):
    l65_to_hb[('numbers', 17, i-35)]=('numbers', 16, i)
for i in range(16, 28+1):
    l65_to_hb[('numbers', 17, i)]=('numbers', 17, i-15)

l65_to_hb[('numbers', 30, 1)] = ('numbers', 29, 40)
for i in range(2, 17+1):
    l65_to_hb[('numbers', 30, i)]=('numbers', 30, i-1)

#psalms
def add_psalms_merging_verse_1_in_1965(chap, maxverse):
    l65_to_hb_merge[(('psalms', chap, 1),('psalms', chap, 2))] = ('psalms', chap, 1)
    for i in range(3, maxverse+1):
        l65_to_hb[('psalms', chap, i)]=('psalms', chap, i-1)
        
add_psalms_merging_verse_1_in_1965(3, 9)
add_psalms_merging_verse_1_in_1965(4, 9)
add_psalms_merging_verse_1_in_1965(5, 13)
add_psalms_merging_verse_1_in_1965(6, 11)
add_psalms_merging_verse_1_in_1965(7, 18)
add_psalms_merging_verse_1_in_1965(8, 10)
add_psalms_merging_verse_1_in_1965(9, 21)
add_psalms_merging_verse_1_in_1965(11, 8)
add_psalms_merging_verse_1_in_1965(12, 9)
add_psalms_merging_verse_1_in_1965(13, 7)
add_psalms_merging_verse_1_in_1965(18, 51)
add_psalms_merging_verse_1_in_1965(19, 15)
add_psalms_merging_verse_1_in_1965(20, 10)
add_psalms_merging_verse_1_in_1965(21, 14)
add_psalms_merging_verse_1_in_1965(22, 32)
add_psalms_merging_verse_1_in_1965(30, 13)
#so far ok, therefore will leave the rest of psalms as is no check
add_psalms_merging_verse_1_in_1965(31, 25)
add_psalms_merging_verse_1_in_1965(34, 23)
add_psalms_merging_verse_1_in_1965(36, 13)
add_psalms_merging_verse_1_in_1965(38, 23)
add_psalms_merging_verse_1_in_1965(39, 14)
add_psalms_merging_verse_1_in_1965(40, 18)
add_psalms_merging_verse_1_in_1965(41, 14)
add_psalms_merging_verse_1_in_1965(42, 12)
add_psalms_merging_verse_1_in_1965(44, 27)
add_psalms_merging_verse_1_in_1965(45, 18)
add_psalms_merging_verse_1_in_1965(46, 12)
add_psalms_merging_verse_1_in_1965(47, 10)
add_psalms_merging_verse_1_in_1965(48, 15)
add_psalms_merging_verse_1_in_1965(49, 21)
add_psalms_merging_verse_1_in_1965(51, 20)
add_psalms_merging_verse_1_in_1965(51, 21)
add_psalms_merging_verse_1_in_1965(52, 10)
add_psalms_merging_verse_1_in_1965(52, 11)
add_psalms_merging_verse_1_in_1965(53, 7)
add_psalms_merging_verse_1_in_1965(54, 8)
add_psalms_merging_verse_1_in_1965(54, 9)
add_psalms_merging_verse_1_in_1965(55, 24)
add_psalms_merging_verse_1_in_1965(56, 14)
add_psalms_merging_verse_1_in_1965(57, 12)
add_psalms_merging_verse_1_in_1965(58, 12)
add_psalms_merging_verse_1_in_1965(59, 18)
add_psalms_merging_verse_1_in_1965(60, 13)
add_psalms_merging_verse_1_in_1965(60, 14)
add_psalms_merging_verse_1_in_1965(61, 9)
add_psalms_merging_verse_1_in_1965(62, 13)
add_psalms_merging_verse_1_in_1965(63, 12)
add_psalms_merging_verse_1_in_1965(64, 11)
add_psalms_merging_verse_1_in_1965(65, 14)
add_psalms_merging_verse_1_in_1965(67, 8)
add_psalms_merging_verse_1_in_1965(68, 36)
add_psalms_merging_verse_1_in_1965(69, 37)
add_psalms_merging_verse_1_in_1965(70, 6)
add_psalms_merging_verse_1_in_1965(75, 11)
add_psalms_merging_verse_1_in_1965(76, 13)
add_psalms_merging_verse_1_in_1965(77, 21)
add_psalms_merging_verse_1_in_1965(80, 20)
add_psalms_merging_verse_1_in_1965(81, 17)
add_psalms_merging_verse_1_in_1965(83, 19)
add_psalms_merging_verse_1_in_1965(84, 13)
add_psalms_merging_verse_1_in_1965(85, 14)
add_psalms_merging_verse_1_in_1965(88, 19)
add_psalms_merging_verse_1_in_1965(89, 53)
add_psalms_merging_verse_1_in_1965(92, 16)
add_psalms_merging_verse_1_in_1965(102, 29)
add_psalms_merging_verse_1_in_1965(108, 14)
add_psalms_merging_verse_1_in_1965(140, 14)
add_psalms_merging_verse_1_in_1965(142, 8)

#song
l65_to_hb_merge[(('songs', 4, 16),('songs', 4, 17))] = ('songs', 4, 16)
l65_to_hb[('songs', 7, 1)] = ('songs', 6, 13)
for i in range(2, 14+1):
    l65_to_hb[('songs', 7, i)]=('songs', 7, i-1)

#zechariah
for i in range(18, 21+1):
    l65_to_hb[('zechariah', 2, i-17)]=('zechariah', 1, i)
for i in range(5, 17+1):
    l65_to_hb[('zechariah', 2, i)]=('zechariah', 2, i-4)
#found out..
l65_to_hb_merge[(('genesis', 14, 20),('genesis', 14, 21))] = ('genesis', 24, 20)
l65_to_hb[('genesis', 14, 22)]=('genesis', 14, 21)
l65_to_hb_merge[(('genesis', 14, 23), "zeme")]=(('genesis', 14, 22), ('genesis', 14, 23))


# ebreju vietturis : lv vietturis
hb_to_l65 = {v: k for k, v in l65_to_hb.items() if v[0]!= '!'}
hb_to_l65_merge = {v: k for k, v in l65_to_hb_merge.items()}
for k, v in l65_to_hb_merge.items():
    #multiple hebrews to single latvian
    if isinstance(v[1], tuple):
        for resolvedVerse in v:
            #this is case where split string is specified for single latvian verse
            if(isinstance(k[0], tuple) and isinstance(k[1], str)):
                hb_to_l65[resolvedVerse] = ("!"+k[0][0], k[0][1], k[0][2])
                l65_to_hb[k[0]] = ("!", -1, -1)
            #also never happens in current list, what was the reasoning putting else here, "just so that it is"?
            else:
                hb_to_l65[resolvedVerse] = ("!"+k[0], k[1], k[2])
                l65_to_hb[k[0]] = ("!", -1, -1)
#value is single hebrew verse. so if key 2nd is tuple not string then
    elif isinstance(k[1], tuple):
        #where to split the hebrew verse (no case actually in current list!
        if(isinstance(v[0], tuple) and isinstance(v[1], str)):
            for resolvedVerse in k:
                hb_to_l65[resolvedVerse] = ("!"+v[0][0], v[0][1], v[0][2])
                hb_to_l65[v[0]] = ("!", -2, -2)
        #single heb verse maps to multiple lvs, so put pointer mark ! (watch merges instead of direct)
        #and the rest two just placeholders so that they are not misused anywhere as misguiding references
        else:
            for resolvedVerse in k:
                l65_to_hb[resolvedVerse] = ("!"+v[0], v[1], v[2])
            #print(v[0])
            hb_to_l65[v] = ("!", -2, -2)
### false alarm, this turns out was standard case of 2 hb verses in single lv verse.
#    elif isinstance(k[1], str) and len(v) == 3:
#        #"part of the verse is one verse in hebrew". for now i will leave the latvian to hebrew map not specified, as its not used for now anyways
#        hb_to_l65[v] = ("!", -2, -2)
#        #print(v)
#!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
#!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!! IZLAISTS PANTS 1965
#!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
hb_to_l65[('exodus', 1, 21)] = None


### helper: json verse key → CATSS verse key

In [ ]:
def catss_key_for_json(book, chap, verse):
    '''
    Translate a JSON (MT) verse key into the CATSS verse key.
    Returns:
      (catss_key, status)
      status: 'identity' | 'remapped' | 'skip_merge' | 'skip_missing'
    'skip_merge' means the verse is part of a merge/split group and this
    notebook will not try to fix it (too ambiguous to reassign greek).
    '''
    k = (book, chap, verse)
    if k not in hb_to_l65:
        return k, 'identity'
    v = hb_to_l65[k]
    if v is None:
        # MT verse that doesn't exist in Latvian/CATSS (e.g. exodus 1:21)
        return None, 'skip_missing'
    if isinstance(v[0], str) and v[0].startswith('!'):
        # placeholder marker from the merge-derivation logic: this verse is
        # part of a merge/split group
        return None, 'skip_merge'
    return v, 'remapped'


## CATSS .par.html parser

Parses a whole-book `.par.html` into
`{(chapter, verse): [cells]}` where each cell captures one alignment row.

Known row types handled:
- **normal**: Hebrew word(s) ↔ Greek word(s).
- **multi-word Hebrew cell**: e.g. `את ראשׁ` — stored as `heb_words` list of
  length >1; greek belongs to the whole group.
- **`---` minus** (`class="minus"`): Greek side empty (LXX omission).
- **`--+` plus**: Hebrew side is `--+`, Greek is an LXX addition with no
  Hebrew counterpart. Flagged `is_plus`; destined for `leftover_greek`.
- **transposition `^`**: cell A has Hebrew word marked `^` and empty Greek;
  a later cell B has empty Hebrew and trans `^ ^^^ =TRANS`. We move cell B's
  Greek onto cell A (the one whose first trans token == `TRANS`).
- **`{...}` particle rows**: empty Hebrew, trans `{...}`, Greek is a particle
  belonging to the *following* real cell. We merge them forward.
- **`=;` switch cases**: retroversion annotation in the trans — the
  heb-native field already holds the MT form, which is what we want; we just
  ignore the retroversion.
- **note rows** (`class="note"`): skipped.


In [ ]:
# Class-aware HTML parser using BeautifulSoup. Robust against:
#   - the original 4-column layout (heb-native, heb-trans, grk-native, grk-trans)
#   - the new 5-column layout with an extra `colb` column for column-b retroversions
#   - inner HTML inside cells (<span>, <br>, <bdi class="ref">, &zwj; etc.)
#   - <tbody> wrappers, missing trailing space in class attributes, etc.
from bs4 import BeautifulSoup

def _td_text(td):
    '''Extract clean text from a <td>, dropping inner UI tags.

    Removes the following UI-only elements before extracting text:
      - <span class="anno">ⓘ</span>          CATSS metadata hover icons
      - <span class="tag …">…</span>          UI labels: "col b", "↶ transp",
                                              "↷ transp", "−LXX", "+LXX"
      - <bdi class="ref">⟨…⟩</bdi>            cross-reference brackets

    None of these carry alignment data; they just decorate the rendered
    table for human readers.
    '''
    if td is None:
        return ''
    # Strip CATSS UI annotation icons
    for anno in td.find_all('span', class_='anno'):
        anno.decompose()
    # Strip UI label tags (col b, transp arrows, ±LXX badges)
    for tag in td.find_all('span', class_='tag'):
        tag.decompose()
    # Strip cross-reference brackets
    for bdi in td.find_all('bdi', class_='ref'):
        bdi.decompose()
    # Use a space separator so <br>, </span> don't run words together,
    # then collapse runs of whitespace.
    txt = td.get_text(separator=' ')
    txt = html_mod.unescape(txt)
    # remove zero-width joiner / non-joiner that mess up tokenizing
    txt = txt.replace('\u200d', '').replace('\u200c', '')
    # collapse whitespace
    txt = re.sub(r'\s+', ' ', txt).strip()
    return txt

def _td_classes(td):
    cls = td.get('class') or []
    return set(cls)

def _find_td(row, *class_names):
    '''Find first <td> in row whose class set contains any of class_names.'''
    for td in row.find_all('td', recursive=False):
        cs = _td_classes(td)
        for cn in class_names:
            if cn in cs:
                return td
    return None

def parse_par_html(path):
    with open(path, 'r', encoding='utf-8') as f:
        soup = BeautifulSoup(f, 'html.parser')

    verses = {}
    for h2 in soup.find_all('h2', class_='verse'):
        ref = h2.get_text(strip=True)
        rm = re.match(r'\S+\s+(\d+):(\d+)', ref)
        if not rm:
            continue
        chap = int(rm.group(1))
        vrs  = int(rm.group(2))

        # The verse table is the next <table class="verse"> sibling
        table = h2.find_next_sibling('table')
        if table is None:
            continue
        cells = _parse_verse_table(table)
        verses[(chap, vrs)] = cells
    return verses

def _parse_verse_table(table):
    cells = []
    for row in table.find_all('tr'):
        row_classes = _td_classes(row) if False else set(row.get('class') or [])
        if 'note' in row_classes:
            continue

        td_heb_native = _find_td(row, 'heb-native')
        td_heb_trans  = _find_td(row, 'heb-trans')
        td_colb       = _find_td(row, 'colb')
        td_grk_native = _find_td(row, 'grk-native')
        td_grk_trans  = _find_td(row, 'grk-trans')

        if td_heb_native is None and td_grk_native is None:
            continue

        heb_native = _td_text(td_heb_native)
        heb_trans  = _td_text(td_heb_trans)
        grk_native = _td_text(td_grk_native)
        grk_trans  = _td_text(td_grk_trans)

        # Detect minus on the grk-native cell. The old 4-column HTML used a
        # 'minus' CSS class; the new 5-column HTML doesn't, so we also detect
        # by content: any cell whose text starts with '--' followed by
        # optional '?' / '?''/ "''" markers and nothing else (or only minus
        # markers + a real Greek word — handled below as inline mixed).
        is_minus_class = (td_grk_native is not None and 'minus' in _td_classes(td_grk_native))
        # tokenize raw greek text first, then test
        _raw_grk_tokens = grk_native.split() if grk_native else []
        # Recognize standalone minus markers: '---', '--', '--?', '---?',
        # combined with optional '?', '??', "''" tokens, with no real Greek.
        def _is_minus_marker(tok):
            t = tok.strip()
            return t in {'---', '--', '--?', '---?', '?', '??', "''", "'"}
        is_minus_content = (bool(_raw_grk_tokens)
                            and all(_is_minus_marker(t) for t in _raw_grk_tokens))
        is_minus = is_minus_class or is_minus_content
        is_plus  = (heb_native == '--+') or heb_trans.startswith('--+')

        # ---- Column-b extraction ----
        # The colb cell contains an alternative Hebrew reading the LXX
        # parent text seems to reflect (e.g. "ידומה .wy" for MT "ודומה").
        # If present and parseable, we keep it as `colb_words` for fallback
        # use when the A-column heb word doesn't match the JSON's MT form.
        colb_text = _td_text(td_colb)
        colb_words = []
        if colb_text:
            # The colb text may contain transliteration markers like
            # ".wy", ".m", "<ge10.4>", "=:", etc. — strip those and keep
            # only the Hebrew word(s).
            colb_clean = colb_text
            # remove =:XXX retroversion markers
            colb_clean = re.sub(r'=:?\S+', '', colb_clean)
            # remove .x suffix annotations (e.g. .wy, .m, .)n)
            colb_clean = re.sub(r'\.\S+', '', colb_clean)
            # remove <...> reference brackets
            colb_clean = re.sub(r'<[^>]*>', '', colb_clean)
            # remove ⟨...⟩ unicode brackets
            colb_clean = re.sub(r'⟨[^⟩]*⟩', '', colb_clean)
            colb_clean = colb_clean.strip()
            if colb_clean:
                colb_words = [w for w in colb_clean.split() if w]

        # CATSS '' marker (per spec): "Long minus or plus (at least four lines)".
        is_long_marker = (heb_native == "''" or heb_trans == "''"
                          or grk_native == "''" or grk_trans == "''")

        # ---- Transposition / sequence markers ----
        # New 5-column HTML uses literal '^^^' as heb_native or grk_native
        # to mark transposition source/target rows. Old 4-column format
        # left these cells empty. We normalize: any cell containing only
        # '^^^' (or similar caret-only content) is treated as empty for
        # purposes of "is this side carrying a real word?".
        def _is_caret_only(s):
            t = s.strip() if s else ''
            return bool(t) and set(t.replace(' ', '')) <= {'^'}

        heb_native_is_caret = _is_caret_only(heb_native)
        grk_native_is_caret = _is_caret_only(grk_native)

        # If heb-native is just '^^^' the row is a transposition TARGET
        # whose anchor word lives elsewhere — treat heb_native as empty
        # for downstream logic (so the existing transpose_target detection
        # via the heb-trans `=XXX` token can find this row).
        if heb_native_is_caret:
            heb_native = ''

        # If grk-native is just '^^^' the row is a transposition SOURCE:
        # the Hebrew word here, but its Greek is moved elsewhere. Treat
        # grk_native as empty so the existing transpose_marker detection
        # picks it up.
        if grk_native_is_caret:
            grk_native = ''

        # transposition markers (after caret normalization)
        transpose_marker = None
        transpose_target = None
        trans_tokens = heb_trans.split()
        if heb_native and (not grk_native) and '^' in trans_tokens:
            transpose_marker = '^'
        if not heb_native and not is_plus:
            for tok in trans_tokens:
                if tok.startswith('='):
                    transpose_target = tok[1:]
                    break

        is_particle = (not heb_native) and ('{...}' in heb_trans)

        # Tokens we strip from heb/grk word lists (not real words):
        #   '?', '??' — CATSS "questionable notation" markers (per spec)
        #   "''", "'" — long-minus/plus block markers
        #   '---', '--' — minus markers (handled by is_minus, but tokens
        #                 may also appear inline mixed with real words)
        #   '^', '^^^' — sequence-difference markers between real words
        #                ('~' / '~~~' in raw CATSS), per spec
        _STRIP_TOKENS = {'?', '??', "''", "'", '---', '--', '--?', '---?',
                         '^', '^^^'}
        def _strip_word_tokens(tokens):
            return [t for t in tokens if t and t not in _STRIP_TOKENS]

        # also normalize tokens like "---?" -> "---" so the inline-minus
        # distribution logic still recognizes them as minuses (we then
        # filter them out with _strip_word_tokens for the final lists).
        def _normalize_minus_token(t):
            if t in ('---?', '--?', '---', '--'):
                return '---'
            return t

        heb_words = []
        if heb_native and heb_native != '--+' and heb_native != "''":
            heb_words = _strip_word_tokens(heb_native.split())

        grk_words_raw = []
        if grk_native and grk_native != "''":
            grk_words_raw = [_normalize_minus_token(w) for w in grk_native.split() if w]
            # strip standalone marker tokens (?, ??, '', ', ^, ^^^)
            grk_words_raw = [w for w in grk_words_raw
                             if w not in {'?', '??', "''", "'", '^', '^^^'}]
        grk_words = [w for w in grk_words_raw if w != '---']

        # Per-word greek for inline --- handling in multi-heb-word cells
        per_word_greek = None
        if len(heb_words) > 1 and '---' in grk_words_raw and len(grk_words_raw) == len(heb_words):
            per_word_greek = []
            for tok in grk_words_raw:
                per_word_greek.append([] if tok == '---' else [tok])
        elif len(heb_words) > 1 and '---' in grk_words_raw:
            groups = []
            current = []
            for tok in grk_words_raw:
                if tok == '---':
                    groups.append(current)
                    current = []
                else:
                    current.append(tok)
            groups.append(current)
            if len(groups) == len(heb_words):
                per_word_greek = groups

        if is_long_marker:
            continue

        cells.append({
            'heb_native': heb_native,
            'heb_trans':  heb_trans,
            'heb_words':  heb_words,
            'colb_words': colb_words,
            'grk_words':  grk_words,
            'per_word_greek': per_word_greek,
            'is_minus':   is_minus,
            'is_plus':    is_plus,
            'is_particle': is_particle,
            'transpose_marker': transpose_marker,
            'transpose_target': transpose_target,
        })

    # Pass 1: resolve ^ transpositions — bidirectional search
    for i, c in enumerate(cells):
        if c['transpose_target'] is None:
            continue
        tgt = c['transpose_target']
        found = False
        for j in range(i - 1, -1, -1):
            cj = cells[j]
            if cj['transpose_marker'] != '^':
                continue
            first_tok = cj['heb_trans'].split()[0] if cj['heb_trans'] else ''
            if first_tok == tgt:
                cj['grk_words'] = cj['grk_words'] + c['grk_words']
                c['grk_words'] = []
                c['_resolved_to'] = j
                found = True
                break
        if found:
            continue
        for j in range(i + 1, len(cells)):
            cj = cells[j]
            if cj['transpose_marker'] != '^':
                continue
            first_tok = cj['heb_trans'].split()[0] if cj['heb_trans'] else ''
            if first_tok == tgt:
                cj['grk_words'] = cj['grk_words'] + c['grk_words']
                c['grk_words'] = []
                c['_resolved_to'] = j
                break

    # Pass 2: merge {...} particle rows forward
    out = []
    pending_particle_grk = []
    for c in cells:
        if c['is_particle']:
            pending_particle_grk.extend(c['grk_words'])
            continue
        if pending_particle_grk and c['heb_words']:
            c['grk_words'] = pending_particle_grk + c['grk_words']
            pending_particle_grk = []
        out.append(c)
    if pending_particle_grk and out:
        out[-1]['grk_words'] = out[-1]['grk_words'] + pending_particle_grk

    return out


## Flatten CATSS cells → per-Hebrew-word expected greek

Each CATSS cell can contain 1..N Hebrew words with a single Greek
group. We flatten such that the first Hebrew word of a multi-word cell
carries the entire Greek group, and the rest carry `[]` — this matches how
our JSON encodes alignments.

Plus cells (`--+`) and any orphan particle greek get collected into
`expected_leftover_greek` for the verse.


In [ ]:
def flatten_cells(cells):
    '''
    Returns:
      flat      : list of dicts {'heb': str, 'heb_alt': [str,...], 'greek': [str,...],
                                  'group_size': int, 'group_first': bool}
                  one per Hebrew word (slash kept intact).
                  - 'heb' is the column-A (MT) form.
                  - 'heb_alt' is the column-B (LXX-parent retroversion) form(s)
                    if any, used as a Hebrew matching fallback.
                  - 'group_size' / 'group_first' as before.
                  - 'greek' is the FULL greek group on the first heb word
                    (or the per-word slice if inline --- was set).
      leftover  : list of str — greek words with no Hebrew anchor
    '''
    flat = []
    leftover = []
    for c in cells:
        if c['is_plus']:
            leftover.extend(c['grk_words'])
            continue
        if not c['heb_words']:
            if c['grk_words']:
                leftover.extend(c['grk_words'])
            continue
        gsize = len(c['heb_words'])
        colb = c.get('colb_words') or []
        # Pair colb words with heb_words positionally if counts match;
        # otherwise share the full colb list as a generic fallback for each.
        def colb_for(idx):
            if colb and len(colb) == gsize:
                return [colb[idx]]
            return list(colb)
        if c.get('per_word_greek') is not None:
            for idx, (hw, gg) in enumerate(zip(c['heb_words'], c['per_word_greek'])):
                flat.append({
                    'heb': hw, 'heb_alt': colb_for(idx), 'greek': list(gg),
                    'group_size': gsize, 'group_first': (idx == 0),
                    'group_inline_split': True,
                })
        else:
            for idx, hw in enumerate(c['heb_words']):
                flat.append({
                    'heb':   hw,
                    'heb_alt': colb_for(idx),
                    'greek': list(c['grk_words']) if idx == 0 else [],
                    'group_size': gsize, 'group_first': (idx == 0),
                    'group_inline_split': False,
                })
    return flat, leftover


## Hebrew normalization for matching

In [ ]:
# Re-defined here so the notebook is self-contained vs 4correct_json.ipynb

def strip_hebrew_for_match(s):
    '''Lowercase-ish Hebrew comparison key: consonants only, no slash,
    collapse final letters to their non-final equivalents so ם==מ etc.'''
    out = []
    FINAL_MAP = {
        '\u05DA': '\u05DB',  # ך → כ
        '\u05DD': '\u05DE',  # ם → מ
        '\u05DF': '\u05E0',  # ן → נ
        '\u05E3': '\u05E4',  # ף → פ
        '\u05E5': '\u05E6',  # ץ → צ
    }
    for c in s:
        if '\u05D0' <= c <= '\u05EA':
            out.append(FINAL_MAP.get(c, c))
    return ''.join(out).replace('/', '')

assert strip_hebrew_for_match('וַיְדַבֵּ֨ר') == strip_hebrew_for_match('ו/ידבר')
assert strip_hebrew_for_match('אֱלֹהִים֒') == strip_hebrew_for_match(strip_hebrew_for_match('אלהים'))
print('hebrew match normalizer ok')


## Diff / fix engine — per verse

Two strategies:

1. **Exact length match** (fast path): flat CATSS length == JSON `hebrew_words`
   length, AND each Hebrew key matches positionally. → replace greek lists
   wholesale, reconcile leftovers.
2. **Per-unique-word safe fixes** (fallback for mismatches): for each distinct
   Hebrew consonant key:
     - CATSS count == JSON count == 1 → safe direct fix.
     - CATSS count == JSON count == N≥2 AND all CATSS greek groups for that
       key are identical → safe bulk fix.
     - otherwise skip (ambiguous position).
   Leftover reconciliation still runs.


In [ ]:
import unicodedata as _ud

def raccs_common(text):
    '''Strip miscellaneous punctuation/quote characters from text before
    comparison. Currently only U+0027 APOSTROPHE is active — used by the
    HTML renderer as a Greek elision mark (e.g. ἀλλ' from beta code ALL').
    Per CATSS Beta Code spec there is NO apostrophe character at all
    (smooth breathing is `)` and rough breathing is `(`), so the elision
    apostrophe is purely an editor's addition. Stripping it lets words
    that differ only in this editor-added mark compare as identical, and
    the JSON's spelling is preserved by `_merge_greek_preserve_case`.

    All other entries are commented out for now; uncomment selectively
    if/when the dataset proves they should also be normalized away.
    '''
    d = {
        #ord('\\N{COMBINING ACUTE ACCENT}'):None,
        ord('\''): None,    # U+0027 APOSTROPHE  ← ACTIVE (editor's elision mark)
        #ord('’'): None,  # U+2019 RIGHT SINGLE QUOTATION MARK
        #ord('‘'): None,  # U+2018 LEFT SINGLE QUOTATION MARK
        #ord('“'): None,  # U+201C LEFT DOUBLE QUOTATION MARK
        #ord('”'): None,  # U+201D RIGHT DOUBLE QUOTATION MARK
        #ord('['): None,
        #ord(']'): None,
        #ord('-'): None,        # HYPHEN-MINUS
        #ord('⧼'): None,   # ⧼ LEFT WHITE ANGLE BRACKET
        #ord('⧽'): None,   # ⧽ RIGHT WHITE ANGLE BRACKET
        #ord('*'): None,
        #ord('⇔'): None,   # ⇔ LEFT RIGHT DOUBLE ARROW
        #ord('〉'): None,   # 〉
        #ord('〈'): None,   # 〈
        #ord('‿'): None,   # ‿ LOW LINE
        #ord('«'): None,   # «
        #ord('»'): None,   # »
        #ord('‹'): None,   # ‹
        #ord('›'): None,   # ›
        #ord('('): None,
        #ord(')'): None,
        #ord(';'): None,
        #ord(','): None,
        #ord('!'): None,
        #ord('?'): None,
        #ord('.'): None,
    }
    return _ud.normalize('NFD', text).translate(d)

def _greek_key(w):
    '''Normalized key for equality checks: lowercased, NFD-normalized,
    combining marks (accents/breathings/iota subscript) stripped, and
    `raccs_common` applied to drop the elision apostrophe (U+0027).
    Two words producing the same key are considered "the same word" and
    the JSON's spelling is preserved by the merge step.'''
    s = raccs_common(w.lower())
    return ''.join(c for c in s if _ud.category(c) != 'Mn')

def _greek_lists_equal(a, b):
    '''Compare greek lists ignoring case, accents, and the elision
    apostrophe (U+0027).'''
    if len(a) != len(b):
        return False
    return all(_greek_key(x) == _greek_key(y) for x, y in zip(a, b))

def _merge_greek_preserve_case(new_from_catss, old_from_json):
    '''
    Produce the new greek list taking CATSS's composition/order,
    but reusing the JSON's original capitalized form whenever the
    normalized key matches something the JSON already had.
    This keeps proper-noun capitalization like 'Ισραηλ', 'Μωυσῆν'.
    '''
    # index old json words by normalized key; preserve first occurrence order
    old_by_key = {}
    for w in old_from_json:
        k = _greek_key(w)
        old_by_key.setdefault(k, w)
    out = []
    for w in new_from_catss:
        k = _greek_key(w)
        out.append(old_by_key.get(k, w))
    return out


def verse_catss_diff(json_verse, catss_cells, key, verbose=True):
    '''
    Compare JSON verse against CATSS cells for one verse.
    Returns dict with:
      status          : 'exact' | 'partial' | 'length_mismatch_skip' | 'no_catss' | 'no_change'
      replaces        : list of (word_idx, old_greek, new_greek)
      leftover_adds   : list[str]
      leftover_removes: list[str]
      new_hebrew_words: list (updated)  — only if any change
      new_leftover    : list[str]       — only if any change
    '''
    result = {
        'status': 'no_catss',
        'replaces': [],
        'leftover_adds': [],
        'leftover_removes': [],
        'new_hebrew_words': None,
        'new_leftover': None,
    }
    if catss_cells is None:
        return result

    flat, catss_leftover = flatten_cells(catss_cells)
    hebrew_words = [dict(w) for w in json_verse.get('hebrew_words', [])]
    leftover_greek = list(json_verse.get('leftover_greek', []))

    # Cleanup: prior buggy runs may have written the literal "''" CATSS
    # long-marker token into greek lists or leftover_greek. Strip those out
    # so the current run can overwrite them cleanly.
    for hw in hebrew_words:
        if hw.get('greek'):
            hw['greek'] = [g for g in hw['greek'] if g != "''"]
    leftover_greek = [g for g in leftover_greek if g != "''"]

    # Precompute the full set of Greek keys CATSS knows for this verse —
    # used by reconcile_leftover when CATSS_AUTHORITATIVE_LEFTOVERS is on.
    catss_all_greek_keys = set()
    for fl in flat:
        for g in fl['greek']:
            catss_all_greek_keys.add(_greek_key(g))
    for g in catss_leftover:
        catss_all_greek_keys.add(_greek_key(g))

    # Detect "long minus" verses: CATSS has hebrew rows but ALL of them
    # have empty greek (the entire stretch is a minus block in the LXX-parent
    # text, often marked with '' in the source .par). In such verses, CATSS
    # has nothing to teach us — the JSON's AI mappings (which probably came
    # from Rahlfs/Göttingen and DO contain the verse) are more authoritative.
    if flat and not catss_leftover and all(not fl['greek'] for fl in flat):
        result['status'] = 'catss_long_minus'
        return result

    # Build a pool of all existing json greek words (mapped + leftover) so
    # we can reuse their original casing for any CATSS word we write back.
    json_full_pool = []
    for hw in hebrew_words:
        json_full_pool.extend(hw.get('greek', []))
    json_full_pool.extend(leftover_greek)

    # ---------- Strategy 1: exact length (with trailing paragraph-marker tolerance) ----------
    PARAGRAPH_MARKERS = {'פ', 'ס'}
    # Trim a single trailing paragraph marker from the JSON side for the
    # purpose of length matching — it's not in CATSS.
    json_effective_len = len(hebrew_words)
    has_trailing_marker = (hebrew_words
        and hebrew_words[-1].get('hebrew', '').strip() in PARAGRAPH_MARKERS)
    if has_trailing_marker:
        json_effective_len -= 1

    if len(flat) == json_effective_len:
        mismatch = False
        for i in range(json_effective_len):
            fl = flat[i]
            hw = hebrew_words[i]
            json_key = strip_hebrew_for_match(hw.get('hebrew', ''))
            if strip_hebrew_for_match(fl['heb']) == json_key:
                continue
            # A-column didn't match — try B-column (colb retroversion)
            if any(strip_hebrew_for_match(alt) == json_key for alt in fl.get('heb_alt', [])):
                continue
            mismatch = True
            break
        if not mismatch:
            i = 0
            while i < json_effective_len:
                fl = flat[i]
                hw = hebrew_words[i]
                gsize = fl.get('group_size', 1)
                inline_split = fl.get('group_inline_split', False)

                if gsize > 1 and not inline_split:
                    # Multi-heb-word CATSS cell with a single greek group.
                    # Check whether the JSON's existing distribution across
                    # these gsize words concatenates to the same greek (case-
                    # insensitively). If yes, leave it alone — JSON's split is
                    # at least as informative. If no, replace with the lump
                    # on the first heb word and clear the rest.
                    catss_group_greek = list(fl['greek'])  # full group greek on first
                    json_concat = []
                    for k in range(gsize):
                        json_concat.extend(hebrew_words[i + k].get('greek', []))
                    if _greek_lists_equal(catss_group_greek, json_concat):
                        # JSON already has correct content; possibly with a
                        # better-than-CATSS distribution. Don't touch it.
                        i += gsize
                        continue
                    # CATSS minus protection: if CATSS has nothing for this
                    # group but the JSON has greek, leave the JSON alone.
                    # CATSS minuses often disagree with Rahlfs/Göttingen which
                    # the JSON's AI mappings are based on.
                    if not catss_group_greek and json_concat:
                        i += gsize
                        continue
                    # else: replace as a lump on the first heb word
                    new_g = _merge_greek_preserve_case(catss_group_greek, json_full_pool)
                    old_g_first = list(hw.get('greek', []))
                    if not _greek_lists_equal(old_g_first, new_g):
                        result['replaces'].append((i, old_g_first, new_g))
                        hebrew_words[i]['greek'] = new_g
                    for k in range(1, gsize):
                        old_g_k = list(hebrew_words[i + k].get('greek', []))
                        if old_g_k:
                            result['replaces'].append((i + k, old_g_k, []))
                            hebrew_words[i + k]['greek'] = []
                    i += gsize
                    continue

                # single-word cell (or inline-split multi-word)
                old_g = list(hw.get('greek', []))
                new_g = _merge_greek_preserve_case(fl['greek'], json_full_pool)
                # CATSS minus protection: don't wipe a non-empty JSON greek
                # with an empty CATSS minus.
                if not new_g and old_g:
                    i += 1
                    continue
                if not _greek_lists_equal(old_g, new_g):
                    result['replaces'].append((i, old_g, new_g))
                    hebrew_words[i]['greek'] = new_g
                i += 1

            adds, removes, new_lo = reconcile_leftover(
                leftover_greek, catss_leftover, hebrew_words, json_full_pool,
                catss_all_greek_keys)
            result['leftover_adds']    = adds
            result['leftover_removes'] = removes
            if result['replaces'] or adds or removes:
                result['new_hebrew_words'] = hebrew_words
                result['new_leftover']     = new_lo
                result['status'] = 'exact'
            else:
                result['status'] = 'no_change'
            return result

    # ---------- Strategy 2: per-unique-word safe fixes ----------
    catss_by_key = defaultdict(list)
    for i, fl in enumerate(flat):
        keys = set()
        ka = strip_hebrew_for_match(fl['heb'])
        if ka:
            keys.add(ka)
        for alt in fl.get('heb_alt', []):
            kb = strip_hebrew_for_match(alt)
            if kb:
                keys.add(kb)
        for k in keys:
            catss_by_key[k].append((i, fl['greek']))

    json_by_key = defaultdict(list)
    for i, hw in enumerate(hebrew_words):
        k = strip_hebrew_for_match(hw.get('hebrew',''))
        if k:
            json_by_key[k].append(i)

    any_fix = False
    for k, catss_hits in catss_by_key.items():
        json_hits = json_by_key.get(k, [])
        if not json_hits:
            continue
        cc = len(catss_hits)
        jc = len(json_hits)

        if cc == 1 and jc == 1:
            ji = json_hits[0]
            new_g = _merge_greek_preserve_case(catss_hits[0][1], json_full_pool)
            old_g = list(hebrew_words[ji].get('greek', []))
            # CATSS minus protection: don't wipe non-empty json with empty catss
            if not new_g and old_g:
                continue
            if not _greek_lists_equal(old_g, new_g):
                result['replaces'].append((ji, old_g, new_g))
                hebrew_words[ji]['greek'] = new_g
                any_fix = True
        elif cc == jc and cc >= 2:
            first = catss_hits[0][1]
            all_same = all(_greek_lists_equal(first, h[1]) for h in catss_hits)
            if all_same:
                new_g = _merge_greek_preserve_case(first, json_full_pool)
                for ji in json_hits:
                    old_g = list(hebrew_words[ji].get('greek', []))
                    if not new_g and old_g:
                        continue
                    if not _greek_lists_equal(old_g, new_g):
                        result['replaces'].append((ji, old_g, new_g))
                        hebrew_words[ji]['greek'] = new_g
                        any_fix = True
            # else: positionally ambiguous, skip

    adds, removes, new_lo = reconcile_leftover(
        leftover_greek, catss_leftover, hebrew_words, json_full_pool,
        catss_all_greek_keys)
    result['leftover_adds']    = adds
    result['leftover_removes'] = removes

    if any_fix or adds or removes:
        result['new_hebrew_words'] = hebrew_words
        result['new_leftover']     = new_lo
        result['status'] = 'partial'
    else:
        result['status'] = 'length_mismatch_skip' if len(flat) != len(hebrew_words) else 'no_change'
    return result


def reconcile_leftover(current_leftover, catss_leftover, hebrew_words, json_full_pool, catss_all_greek_keys):
    '''
    Leftover reconciliation, case-insensitive:
      - Any CATSS leftover greek word whose normalized key isn't already in
        the JSON's full greek pool → added (using preserved case if we've
        seen that key anywhere in the verse).
      - Any JSON leftover greek word whose normalized key matches a mapped
        word in hebrew_words AND isn't in catss_leftover → removed as dup.
      - If CATSS_AUTHORITATIVE_LEFTOVERS is set, any JSON leftover greek
        word whose normalized key doesn't appear in CATSS's full content for
        this verse (mapped + leftover) → removed.  This cleans up residual
        "wrong-verse leftovers" the AI pipeline created in books with LXX/MT
        verse-numbering shifts.
    '''
    mapped_keys = Counter()
    for hw in hebrew_words:
        for g in hw.get('greek', []):
            mapped_keys[_greek_key(g)] += 1
    full_keys = Counter(mapped_keys)
    for g in current_leftover:
        full_keys[_greek_key(g)] += 1

    catss_left_keyed = Counter(_greek_key(w) for w in catss_leftover)

    adds = []
    for w in catss_leftover:
        k = _greek_key(w)
        have = full_keys.get(k, 0)
        want = catss_left_keyed.get(k, 0)
        if have < want:
            preserved = None
            for jw in json_full_pool:
                if _greek_key(jw) == k:
                    preserved = jw
                    break
            adds.append(preserved if preserved is not None else w)
            full_keys[k] = have + 1

    new_leftover = list(current_leftover)
    removes = []
    # If CATSS has *no* Greek content for this verse at all, it cannot
    # authoritatively say which leftover words are "wrong" — disable the
    # authoritative-removal heuristic entirely for that case.
    catss_knows_anything = bool(catss_all_greek_keys)

    for w in list(new_leftover):
        k = _greek_key(w)
        if mapped_keys.get(k, 0) >= 1 and catss_left_keyed.get(k, 0) == 0:
            new_leftover.remove(w)
            removes.append(w)
            continue
        if (CATSS_AUTHORITATIVE_LEFTOVERS and catss_knows_anything
                and k not in catss_all_greek_keys):
            new_leftover.remove(w)
            removes.append(w)

    new_leftover.extend(adds)
    return adds, removes, new_leftover


## Serializer (same compact format as 4correct_json.ipynb)

In [ ]:
def _word_to_compact(w):
    heb     = w.get('hebrew',  '')
    greek   = w.get('greek',   [])
    latvian = w.get('latvian', [])
    hb_str = json.dumps(heb,     ensure_ascii=False)
    gr_str = json.dumps(greek,   ensure_ascii=False)
    lv_str = json.dumps(latvian, ensure_ascii=False)
    return f'{{ "hebrew": {hb_str}, "greek": {gr_str}, "latvian": {lv_str} }}'

def write_condensed_json(path, data):
    verse_blocks = []
    for verse in data:
        words = verse.get('hebrew_words', [])
        compact_words = ',\n      '.join(_word_to_compact(w) for w in words)
        lo_lv = json.dumps(verse.get('leftover_latvian', []), ensure_ascii=False)
        lo_gr = json.dumps(verse.get('leftover_greek',   []), ensure_ascii=False)
        verse_blocks.append(
            f'  {{\n    "hebrew_words": [\n      {compact_words}\n    ],\n'
            f'    "leftover_greek": {lo_gr},\n'
            f'    "leftover_latvian": {lo_lv}\n  }}'
        )
    new_raw = '[\n' + ',\n'.join(verse_blocks) + '\n]'
    with open(path, 'w', encoding='utf-8') as f:
        f.write(new_raw)


## Chapter-level driver

In [ ]:
GREEN = '\033[92m'; RED = '\x1b[38;5;196m'; BLUE = '\x1b[38;5;33m'; DIM = '\x1b[2m'; RESET = '\033[0m'

def process_chapter(bible_book, chapter_num, catss_verses_by_stem, stats, apply_changes=False):
    '''
    catss_verses_by_stem : dict {stem: {(chap, verse): cells}}
      Multiple stems = multiple CATSS traditions for this book (e.g. JoshA+JoshB).
      We apply them in order — second pass only makes safe fixes on top of first.
    '''
    src_path = Path(BIBLE_DIR) / bible_book / f'{chapter_num}.json'
    if not src_path.exists():
        return
    with open(src_path, 'r', encoding='utf-8') as f:
        verses_list = json.load(f)

    any_file_change = False
    for vi, verse_data in enumerate(verses_list):
        verse_num = vi + 1
        # translate MT (json) verse key → CATSS verse key
        remap, rstatus = catss_key_for_json(bible_book, chapter_num, verse_num)
        if rstatus == 'skip_merge':
            stats['status_skip_merge'] += 1
            if verbose_skipped:
                print(f'  {DIM}⏭  {bible_book} {chapter_num}:{verse_num} — merge/split group, not touched{RESET}')
            continue
        if rstatus == 'skip_missing':
            stats['status_skip_missing'] += 1
            continue
        catss_key = (remap[1], remap[2])  # (chap, verse) for catss lookup
        key = catss_key  # for printing consistency

        # accumulate fixes across CATSS traditions
        for stem, catss_map in catss_verses_by_stem.items():
            cells = catss_map.get(catss_key)
            if cells is None:
                continue
            diff = verse_catss_diff(verse_data, cells, (bible_book, chapter_num, verse_num), verbose=verbose)
            stats[f'status_{diff["status"]}'] += 1

            if diff['new_hebrew_words'] is not None:
                verse_data['hebrew_words']  = diff['new_hebrew_words']
                verse_data['leftover_greek'] = diff['new_leftover']
                verses_list[vi] = verse_data
                any_file_change = True

                stats['replaces']         += len(diff['replaces'])
                stats['leftover_adds']    += len(diff['leftover_adds'])
                stats['leftover_removes'] += len(diff['leftover_removes'])

                if verbose and (diff['replaces'] or diff['leftover_adds'] or diff['leftover_removes']):
                    remap_note = f' (←catss {catss_key[0]}:{catss_key[1]})' if rstatus == 'remapped' else ''
                    print(f'  {BLUE}[{stem}]{RESET} {bible_book} {chapter_num}:{verse_num}{remap_note} '
                          f'{diff["status"]} — {len(diff["replaces"])} replace, '
                          f'+{len(diff["leftover_adds"])}/-{len(diff["leftover_removes"])} leftovers')
                    for wi, old, new in diff['replaces'][:6]:
                        h = verse_data['hebrew_words'][wi].get('hebrew','')
                        print(f'    [{wi+1}] {h}  {RED}{old}{RESET} → {GREEN}{new}{RESET}')
                    if len(diff['replaces']) > 6:
                        print(f'    … and {len(diff["replaces"])-6} more')
                    if diff['leftover_adds']:
                        print(f'    {GREEN}+lo{RESET} {diff["leftover_adds"]}')
                    if diff['leftover_removes']:
                        print(f'    {RED}-lo{RESET} {diff["leftover_removes"]}')
            elif diff['status'] == 'length_mismatch_skip' and verbose_skipped:
                flat, _ = flatten_cells(cells)
                print(f'  {DIM}⏭  [{stem}]{RESET} {bible_book} {chapter_num}:{verse_num} '
                      f'length mismatch (catss={len(flat)} json={len(verse_data.get("hebrew_words",[]))}) '
                      f'— no safe word-level fix either')

    if apply_changes and any_file_change:
        write_condensed_json(src_path, verses_list)


## CATSS cache loader

In [ ]:
# Load each .par.html once, keyed by stem
_catss_cache = {}
def load_catss_for_bible(bible_book):
    stems = BIBLE_TO_CATSS.get(bible_book, [])
    result = {}
    for stem in stems:
        if stem not in _catss_cache:
            fp = PAR_DIR_P / f'{stem}.par.html'
            if not fp.exists():
                print(f'⚠️  {fp} not found, skipping')
                _catss_cache[stem] = {}
            else:
                _catss_cache[stem] = parse_par_html(fp)
        result[stem] = _catss_cache[stem]
    return result


## Dry run on a single book (sanity)

In [ ]:
# Flip to a real book name to sanity-check before the full loop
from collections import Counter as _C
_test_stats = _C()
_test_book = 'numbers'
_test_catss = load_catss_for_bible(_test_book)
print(f'{_test_book}: loaded {len(_test_catss)} CATSS tradition(s): {list(_test_catss.keys())}')

_test_book_path = Path(BIBLE_DIR) / _test_book
if _test_book_path.exists():
    _chapters = sorted(int(f.stem) for f in _test_book_path.glob('*.json') if f.stem.isdigit())
    # do just chapter 1 for the dry run
    for ch in _chapters[:1]:
        process_chapter(_test_book, ch, _test_catss, _test_stats, apply_changes=False)
    print()
    print('=== dry-run stats (chapter 1 only, report-only) ===')
    for k, v in sorted(_test_stats.items()):
        print(f'  {k}: {v}')
else:
    print(f'{_test_book_path} does not exist yet — skip dry run')


## Full run — all books

In [ ]:
from collections import Counter
stats = Counter()

books = sorted([d.name for d in Path(BIBLE_DIR).iterdir()
                if d.is_dir() and d.name != 'excel' and d.name=='1_chronicles'])

for book in tqdm(books, desc='🪛 CATSS-fix'):
    if book not in BIBLE_TO_CATSS:
        print(f'⏭  {book}: no CATSS mapping, skipping')
        continue
    catss = load_catss_for_bible(book)
    if not any(catss.values()):
        continue
    book_path = Path(BIBLE_DIR) / book
    chapters = sorted(int(f.stem) for f in book_path.glob('*.json') if f.stem.isdigit())
    if not chapters:
        continue
    for ch in tqdm(chapters, desc=f'📖 {book}', leave=False):
        process_chapter(book, ch, catss, stats, apply_changes=bool(APPLY_CATSS_CHANGES))

print()
print('=' * 60)
print(f'{"APPLIED" if APPLY_CATSS_CHANGES else "REPORT ONLY"} — summary')
print('=' * 60)
for k in sorted(stats):
    print(f'  {k:30s}: {stats[k]}')


## Notes on safety

- **Report mode (`APPLY_CATSS_CHANGES=0`)** never touches disk. Use this
  first to see what would change, fix any surprises, then re-run with `=1`.
- **`CATSS_AUTHORITATIVE_LEFTOVERS=1`** (default) drops any greek word from
  a verse's `leftover_greek` if CATSS doesn't know about that word for that
  verse. This cleans up the residual "wrong-verse leftovers" the prior AI
  pipeline created in books with LXX/MT numbering shifts (1 Samuel 24,
  Numbers 17, Daniel 3-4, Psalms titles, etc.). Set to `0` if you want to
  preserve pre-existing leftovers you've manually curated.
- **Verse-number remapping** uses a verbatim copy of the `l65_to_hb` table
  from `4correct_json.ipynb` cell 7. The Latvian 1965 Bible uses the same
  English verse numbering that CATSS uses, so the Hebrew↔Latvian map also
  works as Hebrew↔CATSS. Merge/split verses (marked with `!` in the
  derivation) are skipped with status `skip_merge` — too ambiguous to
  reassign Greek safely.
- **CATSS UI annotation icons (`ⓘ`)** are stripped before any text
  extraction. The new HTML format wraps every CATSS metadata note
  (`{d}`, `{t}`, `{...}`, `{..d}`, etc.) in `<span class="anno"
  title="..." data-anno="...">ⓘ</span>` for human readers; the parser
  removes these spans so the glyph never bleeds into the data.
- **Question-mark uncertainty markers (`?`, `??`)** per CATSS spec
  ("Questionable notation, equivalent, etc.") are stripped from
  tokenized Hebrew and Greek lists. They are not real words.
- **Questioned-minus variants** (`---?`, `--?`, `--- ?`) are recognized
  as plain minus rows and handled by the same minus protection rule:
  the engine will not wipe a non-empty JSON greek with an empty CATSS
  minus, regardless of whether it's questioned or definite.
- **Minus detection works for both old and new HTML formats.** The old
  4-column files marked minus rows with `class="grk-native minus"`; the
  new 5-column files don't set the CSS class but use `---` text content.
  The parser checks both, so both formats work transparently.
- **HTML parsing uses BeautifulSoup with class-aware lookup**, robust to:
  - the original 4-column layout (`heb-native, heb-trans, grk-native, grk-trans`)
  - the newer 5-column layout that wedges a `colb` column between heb-trans
    and grk-native (containing column-b retroversion text + nested HTML
    markup like `<br>`, `<span style=…>`, `<bdi class="ref">⟨ge10.4⟩</bdi>`)
  - `<tbody>` wrappers, missing trailing space in class attributes,
    `&zwj;` zero-width joiners, and inner tags inside data cells
  Columns are identified by their CSS class name, not by positional index,
  so additional columns won't shift the meaning of any cell.
- **Column-b (LXX-parent retroversion) fallback for Hebrew matching.**
  The new HTML format exposes CATSS column-b — the Hebrew the LXX-parent
  text seems to reflect, often differing from MT (e.g. column-A `ודומה`
  vs column-B `ידומה`). When the column-A Hebrew of a CATSS cell does
  not match the JSON's MT Hebrew, the engine falls back to column-B's
  Hebrew before giving up. This applies to both strategy 1 (positional
  exact-match) and strategy 2 (per-unique-word safe fixes).
- **CATSS minus protection.** When CATSS asserts that a Hebrew word has
  no Greek counterpart (a `---` minus, often part of a long minus block
  marked with `''` in the source `.par`), the engine **never wipes a
  non-empty JSON greek value to `[]`**. CATSS minuses often disagree with
  Rahlfs / Göttingen, which is what the JSON's AI mappings are based on,
  so the JSON's Greek is more authoritative for those cases. Verses where
  *every* CATSS row is a minus get status `catss_long_minus` and are
  skipped entirely. This protects e.g. 1 Chronicles 1:11–24 (the Table of
  Nations names that CATSS marks as minus but Rahlfs has).
- **CATSS-authoritative leftover removal is also disabled** for verses
  where CATSS knows nothing about the verse's Greek content (no mapped
  greek and no leftover greek). It can only remove leftover words when
  CATSS positively asserts something about the verse.
- **Multi-word Hebrew cells preserve existing JSON splits.** When CATSS
  groups N Hebrew words into one alignment cell with a single Greek group
  (e.g. `את נמרוד → τὸν νεβρωδ`), the engine first checks whether the
  JSON's existing per-word distribution across those N words concatenates
  to the same Greek (case-insensitively). If yes, the JSON is left alone
  — its split is at least as informative as the lump. Only when the
  contents differ do we replace, putting the whole Greek group on the
  first Hebrew word.
- **CATSS spec markers handled** per `00.ReadReParallel.txt`:
  `''` (long minus/plus marker) rows are skipped entirely, neither parsed
  as Hebrew nor as Greek words. Any prior-run corruption where `"''"`
  was written into JSON greek lists is auto-cleaned on the next pass.
  `~~~` / `^^^` transposition pointers are resolved **bidirectionally** —
  the moved Greek can appear either before or after its anchor Hebrew word
  in the verse (e.g. 1 Chronicles 4:26 has the `αμουηλ` pointer at row 1
  while its anchor `XMW)L` is at row 4).
- **Multiple CATSS traditions for one bible book** (JoshA+B, JudgesA+B,
  DanielOG+Th) are applied in order. Because the engine only makes
  strictly-safe fixes on length mismatches, a second-tradition pass can
  only correct additional words that the first pass left alone.
- **Hebrew comparison is loose** (consonants only, final-form collapsed,
  slashes removed). This handles pointed MT ↔ unpointed CATSS as well as
  ם/מ etc.
- **Greek comparison is case-and-accent-insensitive**, and when a
  replacement is needed we preserve the JSON's original capitalization
  where possible (so `Ααρων`, `Λευῖται`, `Ισραηλ` stay capitalized even
  though CATSS stores them lowercase).
- **This notebook never edits Hebrew or Latvian fields** — run
  `4correct_json.ipynb` first to get those right, then this one.
